# Chennai MTC RouteRanker - Route Clustering (Module 06)

This notebook implements the complete route-level feature engineering, hyperparameter tuning, and unsupervised clustering pipeline for the Chennai MTC GTFS dataset.

**Strict Integrity Rules Followed:**
- Input datasets are read directly from C:\Users\kavyasri\Downloads\RouteRanker-Git\data\.
- No completed Phase 1 notebooks or datasets are modified.
- All evaluation metrics are derived directly from actual execution without fabrication.


In [1]:
import os
import sys
import re
import warnings
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from math import radians, sin, cos, sqrt, atan2
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

warnings.filterwarnings('ignore')
plt.style.use('ggplot')
print('Environment and libraries initialized successfully.')


Environment and libraries initialized successfully.


In [2]:
# Directory and input path specifications
data_dir = Path(r'C:\Users\kavyasri\Downloads\RouteRanker-Git\data')
project_root = Path(r'C:\Users\kavyasri\Downloads\mtc-gtfs')
out_dir = project_root / 'route_clustering'
reports_dir = project_root / 'reports'
figures_dir = reports_dir / 'figures'

out_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

routes_path = data_dir / 'routes.txt'
trips_path = data_dir / 'trips.txt'
stop_times_path = data_dir / 'stop_times.txt'
stops_path = data_dir / 'stops.txt'
calendar_path = data_dir / 'calendar.txt'
canonical_path = data_dir / 'canonical_routes_mapping.csv'

print(f'Using verified data directory: {data_dir}')
for p in [routes_path, trips_path, stop_times_path, stops_path, calendar_path, canonical_path]:
    assert p.exists(), f'Missing file: {p}'
    print(f'  - {p.name:<30}: {p.stat().st_size:>10,} bytes')


Using verified data directory: C:\Users\kavyasri\Downloads\RouteRanker-Git\data
  - routes.txt                    :    226,405 bytes
  - trips.txt                     :  1,049,101 bytes
  - stop_times.txt                : 43,537,866 bytes
  - stops.txt                     :    230,767 bytes
  - calendar.txt                  :        342 bytes
  - canonical_routes_mapping.csv  :     10,108 bytes


In [3]:
# Load GTFS tables and canonical mapping
t0 = time.time()
routes = pd.read_csv(routes_path, dtype={'route_id': str, 'agency_id': str})
trips = pd.read_csv(trips_path, dtype={'route_id': str, 'trip_id': str, 'service_id': str})
stop_times = pd.read_csv(stop_times_path, dtype={'trip_id': str, 'stop_id': str})
stops = pd.read_csv(stops_path, dtype={'stop_id': str})
calendar = pd.read_csv(calendar_path, dtype={'service_id': str})
canonical = pd.read_csv(canonical_path)

print(f'Routes loaded: {len(routes):,}')
print(f'Trips loaded: {len(trips):,}')
print(f'Stop Times loaded: {len(stop_times):,}')
print(f'Stops loaded: {len(stops):,}')
print(f'Canonical corridors loaded: {len(canonical):,}')


Routes loaded: 4,611
Trips loaded: 47,047
Stop Times loaded: 1,360,443
Stops loaded: 5,580
Canonical corridors loaded: 675


## Route-Level Feature Engineering
Features engineered:
1. stop_count: Unique stops serviced per route.
2. 
oute_length_km: Trajectory distance (km) along representative stop sequence (Haversine).
3. weekday_trip_count & weekend_trip_count: Distinct trips scheduled per service day category.
4. weekday_headway_min & weekend_headway_min: Median interval between scheduled departures.
5. centroid_lat & centroid_lon: Spatial center of mass for stops on the route.
6. Graph topology features: Transit network stop degree and betweenness centrality.
7. Segment traversals: Unique stop-to-stop links and total trip traversal frequency.


In [4]:
# Base route structure and canonical mapping
route_df = routes[['route_id', 'route_short_name', 'route_long_name', 'route_type']].copy()

def extract_canonical(name):
    if not isinstance(name, str):
        return ''
    s = name.strip()
    s = re.sub(r'\s+(CT\d*|CUT\d*|R\d*|RE\d*|E|C\d+)\b', '', s, flags=re.IGNORECASE)
    s = re.sub(r'(?<=[0-9A-Za-z])(CT\d*|CUT\d*)$', '', s, flags=re.IGNORECASE)
    return s.strip().upper()

route_df['canonical_route'] = route_df['route_short_name'].apply(extract_canonical)
route_df = route_df.merge(canonical, on='canonical_route', how='left')

# Trip-stop mapping
trip_stop = trips[['trip_id', 'route_id', 'service_id']].merge(
    stop_times[['trip_id', 'stop_id', 'stop_sequence', 'arrival_time', 'departure_time']],
    on='trip_id'
)

# 1. Stop counts
stop_counts = trip_stop.groupby('route_id')['stop_id'].nunique().reset_index().rename(columns={'stop_id': 'stop_count'})
route_df = route_df.merge(stop_counts, on='route_id', how='left')

# 2. Geometry: Length and Centroids
def haversine_dist(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = radians(lat1), radians(lat2)
    dphi = radians(lat2 - lat1)
    dlambda = radians(lon2 - lon1)
    a = sin(dphi / 2.0) ** 2 + cos(phi1) * cos(phi2) * sin(dlambda / 2.0) ** 2
    return R * 2.0 * atan2(sqrt(a), sqrt(1.0 - a))

stops_geo = stops[['stop_id', 'stop_lat', 'stop_lon']].dropna(subset=['stop_lat', 'stop_lon'])
trip_stop_geo = trip_stop.merge(stops_geo, on='stop_id', how='inner')

trip_lens = trip_stop_geo.groupby(['route_id', 'trip_id'])['stop_sequence'].count().reset_index(name='n_stops')
rep_trips = trip_lens.sort_values(['route_id', 'n_stops'], ascending=[True, False]).drop_duplicates(subset=['route_id'])

trip_geo_sorted = trip_stop_geo.merge(rep_trips[['route_id', 'trip_id']], on=['route_id', 'trip_id']).sort_values(['route_id', 'stop_sequence'])

def calc_geom(group):
    lats = group['stop_lat'].values
    lons = group['stop_lon'].values
    if len(lats) < 2:
        length = 0.0
    else:
        length = sum(haversine_dist(lats[i], lons[i], lats[i+1], lons[i+1]) for i in range(len(lats)-1))
    return pd.Series({
        'route_length_km': length,
        'centroid_lat': float(np.median(lats)) if len(lats) > 0 else np.nan,
        'centroid_lon': float(np.median(lons)) if len(lons) > 0 else np.nan
    })

route_geom = trip_geo_sorted.groupby('route_id').apply(calc_geom).reset_index()
route_df = route_df.merge(route_geom, on='route_id', how='left')

# 3. Temporal Schedules: Weekday vs Weekend Trip Counts and Headways
calendar['is_weekday'] = calendar[['monday', 'tuesday', 'wednesday', 'thursday', 'friday']].max(axis=1) == 1
calendar['is_weekend'] = calendar[['saturday', 'sunday']].max(axis=1) == 1
weekday_services = set(calendar[calendar['is_weekday']]['service_id'])
weekend_services = set(calendar[calendar['is_weekend']]['service_id'])

weekday_trips = trips[trips['service_id'].isin(weekday_services)].groupby('route_id')['trip_id'].nunique().reset_index(name='weekday_trip_count')
weekend_trips = trips[trips['service_id'].isin(weekend_services)].groupby('route_id')['trip_id'].nunique().reset_index(name='weekend_trip_count')

route_df = route_df.merge(weekday_trips, on='route_id', how='left')
route_df = route_df.merge(weekend_trips, on='route_id', how='left')
route_df['weekday_trip_count'] = route_df['weekday_trip_count'].fillna(0).astype(int)
route_df['weekend_trip_count'] = route_df['weekend_trip_count'].fillna(0).astype(int)

def time_to_sec(t):
    try:
        parts = str(t).strip().split(':')
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
    except Exception:
        return np.nan

origin_times = stop_times.sort_values(['trip_id', 'stop_sequence']).groupby('trip_id').first().reset_index()
origin_times['dep_sec'] = origin_times['departure_time'].apply(time_to_sec)
trip_sched = trips[['trip_id', 'route_id', 'service_id']].merge(origin_times[['trip_id', 'dep_sec']], on='trip_id').dropna(subset=['dep_sec'])

def compute_headway_stats(df):
    df_sorted = df.sort_values('dep_sec')
    deps = df_sorted['dep_sec'].values
    if len(deps) < 2:
        return pd.Series({'headway_min': np.nan, 'headway_count': 0})
    diffs = np.diff(deps) / 60.0
    diffs = diffs[diffs > 0]
    if len(diffs) == 0:
        return pd.Series({'headway_min': np.nan, 'headway_count': 0})
    return pd.Series({
        'headway_min': float(np.median(diffs)),
        'headway_count': int(len(diffs))
    })

sched_weekday = trip_sched[trip_sched['service_id'].isin(weekday_services)]
sched_weekend = trip_sched[trip_sched['service_id'].isin(weekend_services)]

hw_weekday = sched_weekday.groupby('route_id').apply(compute_headway_stats).reset_index().rename(
    columns={'headway_min': 'weekday_headway_min', 'headway_count': 'weekday_headway_intervals'}
)
hw_weekend = sched_weekend.groupby('route_id').apply(compute_headway_stats).reset_index().rename(
    columns={'headway_min': 'weekend_headway_min', 'headway_count': 'weekend_headway_intervals'}
)

route_df = route_df.merge(hw_weekday, on='route_id', how='left')
route_df = route_df.merge(hw_weekend, on='route_id', how='left')

# 4. Route-Specific Network Graph Metrics
st_sorted = stop_times.sort_values(['trip_id', 'stop_sequence'])
mask = st_sorted['trip_id'] == st_sorted['trip_id'].shift(-1)
u_nodes = st_sorted['stop_id'][mask].values
v_nodes = st_sorted['stop_id'].shift(-1)[mask].values

edge_df = pd.DataFrame({'u': u_nodes, 'v': v_nodes}).drop_duplicates()
G = nx.from_pandas_edgelist(edge_df, source='u', target='v', create_using=nx.DiGraph())

deg_dict = dict(G.degree())
bc_dict = nx.betweenness_centrality(G, k=min(150, len(G)), seed=42)

stop_metrics = pd.DataFrame({
    'stop_id': list(deg_dict.keys()),
    'node_degree': list(deg_dict.values()),
    'node_betweenness': [bc_dict.get(s, 0.0) for s in deg_dict.keys()]
})

route_stops_unique = trip_stop[['route_id', 'stop_id']].drop_duplicates()
route_stop_net = route_stops_unique.merge(stop_metrics, on='stop_id', how='inner')

route_net_agg = route_stop_net.groupby('route_id').agg({
    'node_degree': 'mean',
    'node_betweenness': 'mean'
}).reset_index().rename(columns={
    'node_degree': 'avg_stop_degree',
    'node_betweenness': 'avg_stop_betweenness'
})

route_df = route_df.merge(route_net_agg, on='route_id', how='left')

trip_edges = pd.DataFrame({
    'trip_id': st_sorted['trip_id'][mask].values,
    'u': u_nodes,
    'v': v_nodes
})
trip_edges['segment'] = trip_edges['u'] + '_' + trip_edges['v']
trip_edges = trip_edges.merge(trips[['trip_id', 'route_id']], on='trip_id')

route_edge_traversals = trip_edges.groupby('route_id').agg(
    route_unique_segments=('segment', 'nunique'),
    total_segment_traversals=('segment', 'count')
).reset_index()

route_df = route_df.merge(route_edge_traversals, on='route_id', how='left')
print(f'Feature engineering successfully completed for {len(route_df):,} routes.')


Feature engineering successfully completed for 4,611 routes.


In [5]:
# Audit missing values and perform principled median imputation
feature_cols = [
    'stop_count',
    'route_length_km',
    'weekday_trip_count',
    'weekend_trip_count',
    'weekday_headway_min',
    'weekend_headway_min',
    'centroid_lat',
    'centroid_lon',
    'avg_stop_degree',
    'avg_stop_betweenness',
    'route_unique_segments',
    'total_segment_traversals'
]

missing_counts = route_df[feature_cols].isna().sum()
missing_pcts = (missing_counts / len(route_df)) * 100.0
missing_report = pd.DataFrame({
    'Feature': feature_cols,
    'Missing_Count': missing_counts.values,
    'Missing_Pct': missing_pcts.values
})
print('=== Missing Value Report ===')
print(missing_report.to_string(index=False))

X = route_df[feature_cols].copy()
imputed_values = {}
for col in feature_cols:
    med = float(X[col].median())
    imputed_values[col] = med
    X[col] = X[col].fillna(med)

print(f'\nTotal missing values after imputation: {X.isna().sum().sum()}')

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Feature matrix scaled: shape {X_scaled.shape}')


=== Missing Value Report ===
                 Feature  Missing_Count  Missing_Pct
              stop_count            677    14.682282
         route_length_km            677    14.682282
      weekday_trip_count              0     0.000000
      weekend_trip_count              0     0.000000
     weekday_headway_min           1546    33.528519
     weekend_headway_min           1546    33.528519
            centroid_lat            677    14.682282
            centroid_lon            677    14.682282
         avg_stop_degree            677    14.682282
    avg_stop_betweenness            677    14.682282
   route_unique_segments            677    14.682282
total_segment_traversals            677    14.682282

Total missing values after imputation: 0
Feature matrix scaled: shape (4611, 12)


In [6]:
# Hyperparameter tuning for K-Means (k=2..6)
k_range = list(range(2, 7))
kmeans_metrics = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    db = davies_bouldin_score(X_scaled, labels)
    ch = calinski_harabasz_score(X_scaled, labels)
    kmeans_metrics.append({'k': k, 'silhouette': sil, 'davies_bouldin': db, 'calinski_harabasz': ch})
    print(f'K-Means (k={k}) -> Silhouette: {sil:.4f}, Davies-Bouldin: {db:.4f}, Calinski-Harabasz: {ch:.1f}')

km_df = pd.DataFrame(kmeans_metrics)
best_k = int(km_df.loc[km_df['silhouette'].idxmax(), 'k'])
print(f'\nOptimal K-Means clusters selected: k = {best_k}')

# Fit final K-Means
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
route_df['kmeans_cluster'] = km_final.fit_predict(X_scaled)
km_sil = silhouette_score(X_scaled, route_df['kmeans_cluster'])
km_db = davies_bouldin_score(X_scaled, route_df['kmeans_cluster'])

# Fit Agglomerative Clustering
agg_final = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
route_df['agg_cluster'] = agg_final.fit_predict(X_scaled)
agg_sil = silhouette_score(X_scaled, route_df['agg_cluster'])
agg_db = davies_bouldin_score(X_scaled, route_df['agg_cluster'])
print(f'Agglomerative (k={best_k}) -> Silhouette: {agg_sil:.4f}, Davies-Bouldin: {agg_db:.4f}')

# DBSCAN Check
dbscan = DBSCAN(eps=1.5, min_samples=10)
route_df['dbscan_cluster'] = dbscan.fit_predict(X_scaled)
n_clusters_db = len(set(route_df['dbscan_cluster'])) - (1 if -1 in route_df['dbscan_cluster'] else 0)
n_noise = (route_df['dbscan_cluster'] == -1).sum()
print(f'DBSCAN check: found {n_clusters_db} clusters and {n_noise} noise points.')


K-Means (k=2) -> Silhouette: 0.4671, Davies-Bouldin: 1.0005, Calinski-Harabasz: 1060.4
K-Means (k=3) -> Silhouette: 0.2686, Davies-Bouldin: 1.4691, Calinski-Harabasz: 1179.1
K-Means (k=4) -> Silhouette: 0.2777, Davies-Bouldin: 1.2821, Calinski-Harabasz: 1215.8
K-Means (k=5) -> Silhouette: 0.3002, Davies-Bouldin: 1.1762, Calinski-Harabasz: 1239.0
K-Means (k=6) -> Silhouette: 0.2748, Davies-Bouldin: 1.2412, Calinski-Harabasz: 1159.3

Optimal K-Means clusters selected: k = 2
Agglomerative (k=2) -> Silhouette: 0.5739, Davies-Bouldin: 0.7142
DBSCAN check: found 2 clusters and 361 noise points.


In [7]:
# Visualisations
fig, ax1 = plt.subplots(figsize=(8, 5))
color = 'tab:blue'
ax1.set_xlabel('Number of Clusters (k)', fontsize=12)
ax1.set_ylabel('Silhouette Score (Higher is better)', color=color, fontsize=12)
ax1.plot(km_df['k'], km_df['silhouette'], marker='o', linewidth=2.5, color=color, label='Silhouette')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Davies-Bouldin Index (Lower is better)', color=color, fontsize=12)
ax2.plot(km_df['k'], km_df['davies_bouldin'], marker='s', linestyle='--', linewidth=2.5, color=color, label='Davies-Bouldin')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('K-Means Cluster Evaluation Metrics across k=2..6', fontsize=14, fontweight='bold', pad=12)
fig.tight_layout()
plt.savefig(figures_dir / 'fig_clustering_silhouette.png', dpi=200)
plt.show()

# Geographic cluster distribution
plt.figure(figsize=(9, 7))
valid_geo = route_df.dropna(subset=['centroid_lat', 'centroid_lon'])
sc = plt.scatter(
    valid_geo['centroid_lon'],
    valid_geo['centroid_lat'],
    c=valid_geo['kmeans_cluster'],
    cmap='viridis',
    s=35,
    alpha=0.75
)
plt.colorbar(sc, label='K-Means Cluster')
plt.title('Chennai MTC Route Spatial Centroids by Cluster', fontsize=13, fontweight='bold', pad=10)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.savefig(figures_dir / 'fig_clustering_route_map.png', dpi=200)
plt.show()

# Cluster profile summary
cluster_summary = route_df.groupby('kmeans_cluster').agg(
    Route_Count=('route_id', 'count'),
    Avg_Stops=('stop_count', 'mean'),
    Avg_Length_km=('route_length_km', 'mean'),
    Avg_Weekday_Trips=('weekday_trip_count', 'mean'),
    Avg_Weekend_Trips=('weekend_trip_count', 'mean'),
    Avg_Weekday_Headway=('weekday_headway_min', 'mean'),
    Avg_Stop_Degree=('avg_stop_degree', 'mean')
).reset_index()
print('=== Cluster Profiles ===')
print(cluster_summary.to_string(index=False))


=== Cluster Profiles ===
 kmeans_cluster  Route_Count  Avg_Stops  Avg_Length_km  Avg_Weekday_Trips  Avg_Weekend_Trips  Avg_Weekday_Headway  Avg_Stop_Degree
              0         4320  23.555861      16.861209           5.432639           5.432639           157.635184         9.368591
              1          291  35.048110      22.477730          81.024055          81.024055            13.269759         8.844619


## Daily Ridership Regression Data Limitation

> **Data Limitation Statement:**  
> The Chennai MTC GTFS specification and existing project datasets provide static timetables, stop locations, and scheduled service frequencies. However, **no empirical daily passenger boardings, tap-in/tap-out smartcard counts, or route-level passenger volume observations exist** in the repository.  
> 
> In accordance with the mentor guidelines and academic integrity standards:
> 1. Synthetic or fabricated passenger volume records are **strictly prohibited**.
> 2. Supervised regression modeling for daily passenger ridership cannot be trained without genuine passenger observations.
> 3. The supply-side clustering established here establishes the operational foundation for route classification, which will directly interface with demand observations once verified ridership data becomes available.
